In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# יצירת pass manager עבור Dynamical Decoupling

<details>
<summary><b>גרסאות חבילות</b></summary>

הקוד בדף זה פותח עם הדרישות הבאות.
אנחנו ממליצים להשתמש בגרסאות האלה או בגרסאות חדשות יותר.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

דף זה מדגים כיצד להשתמש ב-[`PadDynamicalDecoupling`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.PadDynamicalDecoupling) pass כדי להוסיף למעגל טכניקת דיכוי שגיאות הנקראת _dynamical decoupling_.

Dynamical decoupling עובד על ידי הוספת רצפי פולסים (המכונים _רצפי dynamical decoupling_) ל-qubits פנויים כדי לסובב אותם סביב ספירת בלוך, מה שמבטל את השפעת ערוצי הרעש ובכך מדכא דה-קוהרנטיות. רצפי הפולסים האלה דומים לפולסי ריפוקוס המשמשים בתהודה מגנטית גרעינית. לתיאור מלא, ראו את [A Quantum Engineer's Guide to Superconducting Qubits](https://arxiv.org/abs/1904.06560).

מכיוון ש-`PadDynamicalDecoupling` pass פועל רק על מעגלים מתוזמנים וכולל gates שאינם בהכרח basis gates של היעד שלנו, תצטרכו גם את ה-passes [`ALAPScheduleAnalysis`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.ALAPScheduleAnalysis) ו-[`BasisTranslator`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.BasisTranslator).

דוגמה זו משתמשת ב-`ibm_fez`, שאותחל מראש. קבלו את מידע ה-`target` מה-`backend` ושמרו את שמות הפעולות בתור `basis_gates`, כיוון שיהיה צורך לשנות את ה-`target` כדי להוסיף מידע תזמון עבור ה-gates המשמשים ב-dynamical decoupling.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.backend("ibm_fez")

target = backend.target
basis_gates = list(target.operation_names)

צרו מעגל `efficient_su2` כדוגמה. ראשית, בצעו transpile למעגל לפי ה-backend כיוון שפולסי dynamical decoupling צריכים להתווסף לאחר שהמעגל עבר transpilation ותזמון. Dynamical decoupling עובד לרוב הכי טוב כשיש הרבה זמן סרק במעגלים הקוונטיים — כלומר, יש qubits שאינם בשימוש בזמן שאחרים פעילים. זה המצב במעגל הזה כי ה-gates הדו-קוביטיים מסוג `ecr` מוחלים ברצף ב-ansatz זה.

In [2]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.circuit.library import efficient_su2

qc = efficient_su2(12, entanglement="circular", reps=1)
pm = generate_preset_pass_manager(1, target=target, seed_transpiler=12345)
qc_t = pm.run(qc)
qc_t.draw("mpl", fold=-1, idle_wires=False)

<Image src="../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/8228f889-806a-4873-b1da-27c9795d5f5c-0.svg" alt="Output of the previous code cell" />

![פלט של תא הקוד הקודם](../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/8228f889-806a-4873-b1da-27c9795d5f5c-0.svg)

רצף dynamical decoupling הוא סדרה של gates שמרכיבים יחד את הזהות ומרווחים באופן קבוע בזמן. לדוגמה, התחילו בייצור רצף פשוט הנקרא XY4 המורכב מארבעה gates.

In [3]:
from qiskit.circuit.library import XGate, YGate

X = XGate()
Y = YGate()

dd_sequence = [X, Y, X, Y]

בשל התזמון הקבוע של רצפי dynamical decoupling, יש להוסיף מידע על `YGate` ל-`target` כיוון שהוא *אינו* basis gate, בשונה מ-`XGate`. אנחנו יודעים *מראש* ש-`YGate` בעל אותה משך וטעות כמו `XGate`, כך שניתן פשוט לאחזר את אותן תכונות מה-`target` ולהוסיף אותן עבור אובייקטי `YGate`. זו גם הסיבה ש-`basis_gates` נשמרו בנפרד, מכיוון שאנחנו מוסיפים את הוראת `YGate` ל-`target` למרות שהיא אינה basis gate בפועל של `ibm_fez`.

In [4]:
from qiskit.transpiler import InstructionProperties

y_gate_properties = {}
for qubit in range(target.num_qubits):
    y_gate_properties.update(
        {
            (qubit,): InstructionProperties(
                duration=target["x"][(qubit,)].duration,
                error=target["x"][(qubit,)].error,
            )
        }
    )

target.add_instruction(YGate(), y_gate_properties)

מעגלי ansatz כגון `efficient_su2` הם פרמטריים, לכן יש לקשור להם ערכים לפני שליחתם ל-backend. כאן נשייך פרמטרים אקראיים.

In [5]:
import numpy as np

rng = np.random.default_rng(1234)
qc_t.assign_parameters(
    rng.uniform(-np.pi, np.pi, qc_t.num_parameters), inplace=True
)

בשלב הבא, הפעילו את ה-passes המותאמים אישית. אתחלו את `PassManager` עם `ALAPScheduleAnalysis` ו-`PadDynamicalDecoupling`. הריצו קודם את `ALAPScheduleAnalysis` כדי להוסיף מידע תזמון על המעגל הקוונטי, לפני שניתן להוסיף את רצפי dynamical decoupling המרווחים באופן קבוע. ה-passes האלה מורצים על המעגל עם `.run()`.

In [6]:
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes.scheduling import (
    ALAPScheduleAnalysis,
    PadDynamicalDecoupling,
)

dd_pm = PassManager(
    [
        ALAPScheduleAnalysis(target=target),
        PadDynamicalDecoupling(target=target, dd_sequence=dd_sequence),
    ]
)
qc_dd = dd_pm.run(qc_t)

השתמשו בכלי הוויזואליזציה [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) כדי לראות את תזמון המעגל ולאשר שרצף מרווח באופן קבוע של אובייקטי `XGate` ו-`YGate` מופיע במעגל.

In [7]:
from qiskit.visualization import timeline_drawer

timeline_drawer(qc_dd, idle_wires=False, target=target)

<Image src="../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/cb73e2c4-ab05-4f15-91ae-2fab64028d6e-0.svg" alt="Output of the previous code cell" />

![פלט של תא הקוד הקודם](../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/cb73e2c4-ab05-4f15-91ae-2fab64028d6e-0.svg)

לבסוף, מכיוון ש-`YGate` אינו basis gate בפועל של ה-backend שלנו, יש להחיל ידנית את ה-`BasisTranslator` pass (זהו pass ברירת מחדל, אך הוא מורץ לפני התזמון, לכן צריך להחילו שוב). ספריית השוויוניות של הסשן היא ספרייה של שוויוניות מעגלים המאפשרת ל-Transpiler לפרק מעגלים ל-basis gates, כפי שמצוין גם כארגומנט.

In [8]:
from qiskit.circuit.equivalence_library import (
    SessionEquivalenceLibrary as sel,
)
from qiskit.transpiler.passes import BasisTranslator

qc_dd = BasisTranslator(sel, basis_gates)(qc_dd)
qc_dd.draw("mpl", fold=-1, idle_wires=False)

<Image src="../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/aaa27ee4-1965-41bf-abd2-1d9176af6dc4-0.svg" alt="Output of the previous code cell" />

![פלט של תא הקוד הקודם](../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/aaa27ee4-1965-41bf-abd2-1d9176af6dc4-0.svg)

כעת, אובייקטי `YGate` אינם נמצאים במעגל שלנו, ויש מידע תזמון מפורש בצורת gates מסוג `Delay`. המעגל המתורגם עם dynamical decoupling מוכן כעת לשליחה ל-backend.

## צעדים הבאים

> **Tip:** - כדי ללמוד כיצד להשתמש בפונקציה `generate_preset_passmanager` במקום לכתוב passes משלכם, התחילו עם הנושא [הגדרות ברירת מחדל ואפשרויות תצורה לTranspilation](defaults-and-configuration-options).
>     - נסו את המדריך [השוואת הגדרות Transpiler](/guides/circuit-transpilation-settings#compare-transpiler-settings).
>     - ראו את [תיעוד ה-API של Transpile.](https://docs.quantum-computing.ibm.com/api/qiskit/transpiler)